[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C72_MultiView_Geometry_Course/01_camera/01_camera_model.ipynb)

# 01 · 相机模型与畸变

三件事：

1. **把投影拆成三步**，看清「不可逆的那一步不引入误差、引入误差的那两步都可逆」。
2. **量出畸变的边缘性**：10% 对角处 0.26 px、画幅角 197.71 px（**750 倍**），
   以及去畸变迭代的收敛速度**强烈依赖半径**。
3. **把 `K` 在 resize / crop / letterbox 下的变换写对**，
   并给「忘记同步」的四种错法分类——
   **其中只有一种是静默的，而它恰好最危险。**

## 0 · 环境与参数

In [ ]:
import numpy as np

print('numpy', np.__version__)

H  = 1.5
F  = 1200.0
W, HGT = 1920, 1080
CX, CY = W / 2, HGT / 2
K = np.array([[F, 0, CX], [0, F, CY], [0, 0, 1.]])

# 一组典型的车载广角前视畸变系数（桶形，k1<0）
K1, K2, K3 = -0.28, 0.09, -0.012
P1, P2     =  0.0012, -0.0008
DIST = (K1, K2, K3, P1, P2)

R_MAX = np.hypot(CX, CY) / F          # 画幅角对应的归一化半径
print(f'画幅角的归一化半径 r_max = {R_MAX:.4f}')
print(f'水平 FOV = {np.rad2deg(2*np.arctan(CX/F)):.1f}°')

## 1 · 三步投影的分解

拆开写一遍，是为了能分别检查每一步。
断言部分证明「矩阵一行写完」与「三步拆开」完全等价。

In [ ]:
def R_vc(pitch_deg=0.0):
    t = np.deg2rad(pitch_deg)
    return np.array([[0., -np.sin(t),  np.cos(t)],
                     [-1.,        0.,        0.],
                     [0., -np.cos(t), -np.sin(t)]])

CAM_T = np.array([0., 0., H])

def distort(x, y, dist=DIST):
    k1, k2, k3, p1, p2 = dist
    r2 = x * x + y * y
    rad = 1 + k1 * r2 + k2 * r2 ** 2 + k3 * r2 ** 3
    xd = x * rad + 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
    yd = y * rad + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
    return xd, yd

def project_steps(P_v, pitch=0.0, dist=DIST, Kmat=None):
    Kmat = K if Kmat is None else Kmat
    P_c = R_vc(pitch).T @ (np.asarray(P_v, float) - CAM_T)     # ① 刚体
    if P_c[2] <= 1e-9:
        return None
    x, y = P_c[0] / P_c[2], P_c[1] / P_c[2]                    # ② 透视除法
    xd, yd = distort(x, y, dist)                               # ③a 畸变
    uv = Kmat @ np.array([xd, yd, 1.0])                        # ③b 内参
    return {'P_c': P_c, 'norm': (x, y), 'norm_d': (xd, yd), 'uv': uv[:2]}

# 等价性：无畸变时，三步 == 一次矩阵乘 + 齐次除
for P in [(10., 1.75, 0.), (30., -3.2, 2.2), (60., 0., 0.5)]:
    st = project_steps(P, dist=(0, 0, 0, 0, 0))
    Pc = st['P_c']
    uv_mat = (K @ Pc)[:2] / (K @ Pc)[2]
    assert np.allclose(st['uv'], uv_mat, atol=1e-12), (st['uv'], uv_mat)
print('✅ 无畸变时「三步拆开」与「K @ P_c 再齐次除」完全一致')

st = project_steps((30., -3.2, 2.2))
print('\n一个例子（30m 处、离地 2.2m 的限速牌）：')
print(f"  ① 相机坐标   P_c = {np.round(st['P_c'], 4)}")
print(f"  ② 归一化     (x,y) = {tuple(round(v,5) for v in st['norm'])}")
print(f"  ③a 加畸变    (xd,yd) = {tuple(round(v,5) for v in st['norm_d'])}")
print(f"  ③b 到像素    (u,v) = {np.round(st['uv'], 2)}")
print(f"  畸变造成的像素位移 = "
      f"{np.hypot(*(np.array(st['norm_d'])-np.array(st['norm'])))*F:.2f} px")

## 2 · 内参各项的消融

把 `f`、`c_y`、`f_x/f_y` 各扰动一次，看它们分别破坏什么。
**第三行是关键**：主点移动与相机低头对地面点的成像位置有相同的影响。

In [ ]:
def ground_range(v, f=F, cy=CY, h=H):
    '''由像素行反解地面点纵向距离（无畸变、pitch=0）。'''
    den = v - cy
    return np.inf if abs(den) < 1e-9 else h * f / den

v_10m = CY + F * H / 10.0
print(f'10m 地面点成像在 v = {v_10m:.1f}\n')

print(f"{'扰动':30s} {'读出距离':>10s} {'相对误差':>10s}")
for name, f, cy in [('基线', F, CY),
                    ('f 大 10%', F * 1.1, CY),
                    ('f 小 10%', F * 0.9, CY),
                    ('c_y 偏 +21 px', F, CY + 21),
                    ('c_y 偏 -21 px', F, CY - 21)]:
    d = ground_range(v_10m, f, cy)
    print(f'{name:30s} {d:9.3f}m {100*(d-10)/10:9.1f}%')

# f 的误差直接等比例传到距离
assert abs(ground_range(v_10m, F * 1.1, CY) - 11.0) < 1e-9
assert abs(ground_range(v_10m, F * 0.9, CY) -  9.0) < 1e-9
print('\n✅ f 的相对误差 = 距离的相对误差（精确等比例）')

# c_y 与 pitch 的等价性
dcy = 21.0
equiv_deg = np.rad2deg(np.arctan(dcy / F))
d_by_cy    = ground_range(v_10m, F, CY + dcy)
d_by_pitch = H * F / (v_10m - (CY + F * np.tan(np.deg2rad(equiv_deg))))
print(f'主点下移 {dcy:.0f} px  -> 读出 {d_by_cy:.4f} m')
print(f'相机低头 {equiv_deg:.4f}° -> 读出 {d_by_pitch:.4f} m')
assert abs(d_by_cy - d_by_pitch) < 1e-9
print(f'✅ 二者精确等价：f={F:.0f} 时 **1° ≈ {F*np.tan(np.deg2rad(1)):.1f} 个像素**')

## 3 · 畸变是纯边缘现象

In [ ]:
print(f"{'占对角线半长':>12s} {'r':>8s} {'位移 (px)':>12s}")
disp = {}
for frac in [0.10, 0.25, 0.50, 0.75, 0.90, 1.00]:
    x, y = frac * CX / F, frac * CY / F
    xd, yd = distort(x, y)
    d_px = np.hypot(xd - x, yd - y) * F
    disp[frac] = d_px
    print(f'{frac*100:11.0f}% {np.hypot(x,y):8.3f} {d_px:12.2f}')

ratio = disp[1.00] / disp[0.10]
print(f'\n画幅角 / 10% 处 = **{ratio:.0f} 倍**')
assert ratio > 500, '畸变应当是强烈的边缘现象'

# 位移大致按 r^3 走（主导项 k1*r^2 再乘 r）
rs = np.array([np.hypot(f*CX/F, f*CY/F) for f in disp])
ds = np.array(list(disp.values()))
expo = np.polyfit(np.log(rs), np.log(ds), 1)[0]
print(f'log-log 斜率 = {expo:.2f}  → 位移 ≈ r^{expo:.1f}（理论主导项是 r³）')
assert 2.5 < expo < 3.2, f'指数应接近 3，实测 {expo:.2f}'
print('✅ 畸变位移随半径按约三次幂增长')

## 4 · 去畸变的不动点迭代：收敛速度依赖半径

**注意最后一行**：在画幅角上，`n=5`（很多实现的默认值）之后还剩 0.6 px。

In [ ]:
def undistort_iter(xd, yd, n, dist=DIST):
    k1, k2, k3, p1, p2 = dist
    x, y = xd, yd
    for _ in range(n):
        r2 = x * x + y * y
        rad = 1 + k1 * r2 + k2 * r2 ** 2 + k3 * r2 ** 3
        dx = 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
        dy = p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
        x, y = (xd - dx) / rad, (yd - dy) / rad
    return x, y

NS = [1, 2, 3, 5, 8, 12]
print('残差（像素）：')
print(f"{'位置':>10s} " + ''.join(f'n={n}'.rjust(11) for n in NS))
resid = {}
for frac in [0.25, 0.50, 0.75, 1.00]:
    x0, y0 = frac * CX / F, frac * CY / F
    xd, yd = distort(x0, y0)
    row = []
    for n in NS:
        xu, yu = undistort_iter(xd, yd, n)
        row.append(np.hypot(xu - x0, yu - y0) * F)
    resid[frac] = row
    print(f'{frac*100:9.0f}% ' + ''.join(f'{e:11.2e}' for e in row))

# 同一个 n，残差随半径单调上升
for j in range(len(NS)):
    col = [resid[f][j] for f in [0.25, 0.50, 0.75, 1.00]]
    assert col == sorted(col), f'n={NS[j]} 时残差应随半径上升'

e5_corner = resid[1.00][NS.index(5)]
print(f'\nn=5 在画幅角上的残差 = {e5_corner:.3f} px')
assert e5_corner > 0.3, '这就是「默认迭代 5 次」的代价'

# 换算成米：用 50m 处的 1px 灵敏度
v50 = CY + F * H / 50.0
m_per_px = abs(ground_range(v50 + 1) - 50.0)
print(f'50m 处 1 px = {m_per_px:.2f} m  →  {e5_corner:.3f} px ≈ '
      f'**{e5_corner*m_per_px:.2f} m**')
print('✅ 一个「默认参数」在画幅角上值接近一米')

## 5 · `K` 在预处理下的变换，与四种错法

每个改变图像几何的操作都是一次左乘 $S$。**「部分正确」比「全错」危险。**

In [ ]:
def S_mat(sx, sy, tx, ty):
    return np.array([[sx, 0, tx], [0, sy, ty], [0, 0, 1.]])

OPS = {
    'resize 到一半':          S_mat(0.5, 0.5, 0, 0),
    'center-crop 到 1280x720': S_mat(1, 1, -(W-1280)/2, -(HGT-720)/2),
    'letterbox 到 640x640':    S_mat(640/W, 640/W, 0, (640 - HGT*640/W)/2),
}
for name, S in OPS.items():
    Kp = S @ K
    print(f'{name:26s} f=({Kp[0,0]:7.2f},{Kp[1,1]:7.2f})  c=({Kp[0,2]:7.2f},{Kp[1,2]:7.2f})')

# letterbox 的 padding 量自查
pad = (640 - HGT * 640 / W) / 2
print(f'\nletterbox 上下各 padding {pad:.0f} px')

print('\n—— 图像缩放到一半后，10m 地面点的四种读法 ——')
v_half = v_10m * 0.5
cases = [('全对',              600., 270.),
         ('两个都忘了改',      1200., 540.),
         ('只改焦距、忘主点',   600., 540.),
         ('只改主点、忘焦距',  1200., 270.)]
reads = {}
for name, f, cy in cases:
    d = ground_range(v_half, f, cy)
    reads[name] = d
    loud = '响的（负距离）' if d < 0 else ('—' if abs(d-10) < 1e-9 else '**静默**')
    print(f'  {name:20s} f={f:6.0f} c_y={cy:5.0f} -> {d:8.2f} m   {loud}')

assert abs(reads['全对'] - 10.0) < 1e-9
assert reads['两个都忘了改'] < 0 and reads['只改焦距、忘主点'] < 0
assert abs(reads['只改主点、忘焦距'] - 20.0) < 1e-9
print('\n✅ 三种错法里两种给负距离（会被 sanity check 抓住），'
      '而「只改主点」静默地错 2 倍')

## 6 · 畸变模型的单调性检查

$r_d(r)$ 必须在整个画幅内单调递增。一旦折返，说明多项式在画幅内已经发散——
**同一个畸变后半径对应两个原始半径，去畸变无解。**

In [ ]:
def rd_of_r(r, dist):
    k1, k2, k3 = dist[:3]
    r2 = r * r
    return r * (1 + k1 * r2 + k2 * r2 ** 2 + k3 * r2 ** 3)

def first_fold(dist, r_hi=3.0, n=8001):
    '''返回 r_d(r) 首次不再递增的半径；在 [0, r_hi] 内全程单调则返回 None。

    注意 r_hi 会影响结论：几乎所有多项式最终都会折返，
    所以有意义的量不是「有没有折返」，而是**折返点相对画幅角的余量**。
    '''
    r = np.linspace(0, r_hi, n)
    bad = np.where(np.diff(rd_of_r(r, dist)) <= 0)[0]
    return None if len(bad) == 0 else float(r[bad[0]])

SETS = {
    '本课系数':            DIST,
    '只有 k1=-0.28':       (-0.28, 0, 0, 0, 0),
    '激进广角':            (-0.55, 0.30, 0.0, 0, 0),
    '过拟合的三系数':      (-0.40, 0.60, -1.20, 0, 0),
}
print(f'画幅角 r_max = {R_MAX:.4f}   (余量 = 折返半径 / r_max)')
print()
folds = {}
for name, d in SETS.items():
    fold = first_fold(d)
    folds[name] = fold
    if fold is None:
        print(f'  OK   {name:18s} 到 r=3.0 都不折返，余量 = inf')
    else:
        m = fold / R_MAX
        flag = 'OK  ' if m >= 1.2 else ('WARN' if m > 1.0 else 'FAIL')
        note = '在画幅外' if m > 1.0 else '**在画幅内**'
        print(f'  {flag} {name:18s} 折返于 r={fold:.4f}  余量={m:5.2f}  {note}')

# 本课系数确实会折返，只是折返点远在画幅之外 —— 这才是准确的说法
assert 1.8 < folds['本课系数'] < 1.9, folds['本课系数']
assert folds['本课系数'] / R_MAX > 2.0, '本课系数的余量应超过 2 倍'
assert folds['只有 k1=-0.28'] > R_MAX, '单 k1 的折返点仍在画幅外'
assert folds['只有 k1=-0.28'] / R_MAX < 1.2, '但它的余量不足 1.2，应当报警'
assert folds['过拟合的三系数'] < R_MAX, '过拟合系数应在画幅内折返'
assert folds['激进广角'] is None
print()
print('✅ 余量把四组系数分成三档：安全(2.03 / inf) · '
      '**余量不足(1.19)** · 画幅内发散(0.79)')
print('   -> 「有没有折返」是个错问题（几乎都会折返），'
      '**「余量够不够」才是可门禁的量**')

## 7 · 小结

| 结论 | 数值 |
|---|---|
| 三步里只有透视除法不可逆，而它不引入误差 | 等价性误差 < 1e-12 |
| $f$ 的相对误差 = 距离的相对误差 | 精确等比例 |
| 主点 ↔ pitch 等价 | $f=1200$ 时 **1° ≈ 20.9 px** |
| 畸变是边缘现象 | 0.26 → 197.71 px，**750 倍**，$\propto r^{3}$ |
| 去畸变 `n=5` 在画幅角 | 残差 0.60 px ≈ **0.81 m @ 50m** |
| 四种 `K` 错法 | 两种给负距离，**一种静默地错 2 倍** |
| 单调性检查 | 余量分三档：**2.03 / inf** 安全 · **1.19** 不足 · **0.79** 画幅内发散 |

## ✏️ 练习 1：按半径自适应的去畸变

固定迭代次数在中心是浪费、在边缘不够。

实现 `undistort_adaptive(xd, yd, tol_px=1e-3, max_iter=30)`：
迭代直到**相邻两轮的位移小于 `tol_px` 个像素**或达到 `max_iter`，
返回 `(x, y, n_used)`。

In [ ]:
def undistort_adaptive(xd, yd, tol_px=1e-3, max_iter=30, dist=DIST):
    """迭代到收敛。返回 (x, y, 实际用的轮数)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
FRACS = [0.10, 0.25, 0.50, 0.75, 1.00]
truth = {}
for fr in FRACS:
    x0, y0 = fr * CX / F, fr * CY / F
    truth[fr] = (x0, y0, *distort(x0, y0))

print(f"{'位置':>8s} {'用的轮数':>9s} {'残差 (px)':>12s}")
used = []
for fr in FRACS:
    x0, y0, xd, yd = truth[fr]
    x, y, n = undistort_adaptive(xd, yd)
    err = np.hypot(x - x0, y - y0) * F
    used.append(n)
    print(f'{fr*100:7.0f}% {n:9d} {err:12.2e}')
    assert err < 1e-2, f'{fr} 处残差 {err:.3e} px 过大'

assert used == sorted(used), '边缘应当比中心用更多轮'
assert used[0] <= 3, '中心几轮就该够'
assert used[-1] > used[0], '画幅角必须用更多轮'
tot_fixed = 12 * len(FRACS)
print(f'\n自适应总轮数 {sum(used)} vs 固定 n=12 的 {tot_fixed} 轮'
      f'  → 省了 {100*(1-sum(used)/tot_fixed):.0f}%')
print('✅ 练习 1 通过：精度由容差保证，而不是由一个隐藏的迭代次数保证')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def undistort_adaptive(xd, yd, tol_px=1e-3, max_iter=30, dist=DIST):
    k1, k2, k3, p1, p2 = dist
    x, y = xd, yd
    for n in range(1, max_iter + 1):
        r2 = x * x + y * y
        rad = 1 + k1 * r2 + k2 * r2 ** 2 + k3 * r2 ** 3
        dx = 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
        dy = p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
        xn, yn = (xd - dx) / rad, (yd - dy) / rad
        step = np.hypot(xn - x, yn - y) * F
        x, y = xn, yn
        if step < tol_px:
            return x, y, n
    return x, y, max_iter

for fr in FRACS:
    x0, y0, xd, yd = truth[fr]
    x, y, n = undistort_adaptive(xd, yd)
    assert np.hypot(x - x0, y - y0) * F < 1e-2
print('✅ 参考答案 1 通过（判据用「相邻两轮的位移」而不是「残差」——'
      '因为真值在运行时是不知道的）')

## ✏️ 练习 2：把预处理写成 `K` 的变换

实现 `adjust_K(K, op, **kw)`，支持三种操作并返回 `(K', 输出尺寸)`：

- `'resize'` —— `kw: out_w, out_h`（允许非等比）
- `'crop'` —— `kw: x0, y0, out_w, out_h`
- `'letterbox'` —— `kw: side`（缩放到长边等于 `side`，短边居中 padding）

要求：三种都通过左乘一个 $S$ 实现，不要各写一套公式。

In [ ]:
def adjust_K(Kmat, op, in_w=W, in_h=HGT, **kw):
    """返回 (K', (out_w, out_h))。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
K_half, sz = adjust_K(K, 'resize', out_w=960, out_h=540)
assert sz == (960, 540)
assert np.allclose([K_half[0,0], K_half[1,1]], [600, 600])
assert np.allclose([K_half[0,2], K_half[1,2]], [480, 270])

K_crop, sz = adjust_K(K, 'crop', x0=320, y0=180, out_w=1280, out_h=720)
assert sz == (1280, 720)
assert np.allclose([K_crop[0,0], K_crop[1,1]], [F, F]), 'crop 不改焦距'
assert np.allclose([K_crop[0,2], K_crop[1,2]], [640, 360])

K_lb, sz = adjust_K(K, 'letterbox', side=640)
assert sz == (640, 640)
s = 640 / W
assert np.allclose([K_lb[0,0], K_lb[1,1]], [F*s, F*s])
assert np.allclose(K_lb[0,2], CX*s)
assert np.allclose(K_lb[1,2], CY*s + (640 - HGT*s)/2)

# 非等比 resize 会让 f_x != f_y —— 这是第 2 节说的「比值应在 1±0.002」的破坏源
K_sq, _ = adjust_K(K, 'resize', out_w=640, out_h=640)
assert abs(K_sq[0,0]/K_sq[1,1] - (640/W)/(640/HGT)) < 1e-12
print(f'非等比 resize 到 640x640: f_x/f_y = {K_sq[0,0]/K_sq[1,1]:.4f} '
      '（圆牌会被读成椭圆）')
print('✅ 练习 2 通过：三种操作都是一次左乘')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def adjust_K(Kmat, op, in_w=W, in_h=HGT, **kw):
    if op == 'resize':
        ow, oh = kw['out_w'], kw['out_h']
        S = S_mat(ow / in_w, oh / in_h, 0, 0)
    elif op == 'crop':
        ow, oh = kw['out_w'], kw['out_h']
        S = S_mat(1, 1, -kw['x0'], -kw['y0'])
    elif op == 'letterbox':
        side = kw['side']
        s = side / max(in_w, in_h)
        S = S_mat(s, s, (side - in_w * s) / 2, (side - in_h * s) / 2)
        ow = oh = side
    else:
        raise ValueError(op)
    return S @ Kmat, (int(ow), int(oh))

K_lb, sz = adjust_K(K, 'letterbox', side=640)
assert sz == (640, 640) and np.allclose(K_lb[1,2], CY*640/W + (640 - HGT*640/W)/2)
print('✅ 参考答案 2 通过')

## ✏️ 练习 3：畸变模型的越界检查

实现 `distortion_audit(dist, r_max)`，返回 dict：

- `'monotonic'` —— bool：$r_d(r)$ 在 $[0, r_{max}]$ 上是否单调递增
- `'fold_at'` —— float 或 None：首次折返的半径（**可以大于 $r_{max}$**）
- `'margin'` —— float：`fold_at / r_max`（无折返时为 `inf`）
- `'max_incidence_deg'` —— float：$r_{max}$ 对应的入射角 $\arctan r_{max}$

判据：`monotonic` 为假、或 `margin < 1.2`、或入射角 > 75°，都应当报警。

> `monotonic` 与 `margin` 是**两个不同的信号**：
> 单 $k_1$ 那组在画幅内单调（`monotonic=True`），但余量只有 1.19 —— **它该报警**。

In [ ]:
def distortion_audit(dist, r_max):
    """返回 dict(monotonic, fold_at, margin, max_incidence_deg)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
a_ok  = distortion_audit(DIST, R_MAX)
a_bad = distortion_audit((-0.40, 0.60, -1.20, 0, 0), R_MAX)
a_thin = distortion_audit((-0.28, 0, 0, 0, 0), R_MAX)

assert set(a_ok) == {'monotonic', 'fold_at', 'margin', 'max_incidence_deg'}
assert a_ok['monotonic'] is True, '本课系数在画幅内单调'
assert a_ok['margin'] > 2.0, '本课系数的余量应超过 2 倍'
assert a_bad['monotonic'] is False
assert a_bad['fold_at'] < R_MAX, '过拟合系数应在画幅内折返'
assert a_thin['monotonic'] is True and 1.0 < a_thin['margin'] < 1.2,     '单 k1 在画幅内单调，但余量不足 1.2 —— 两个信号必须分开'
assert abs(a_ok['max_incidence_deg'] - np.rad2deg(np.arctan(R_MAX))) < 1e-9
assert a_bad['margin'] < 1.0
assert distortion_audit((-0.55, 0.30, 0.0, 0, 0), R_MAX)['margin'] == float('inf')

for name, a in [('本课系数', a_ok), ('只有 k1', a_thin), ('过拟合三系数', a_bad)]:
    fold = 'None' if a['fold_at'] is None else f"{a['fold_at']:.4f}"
    marg = 'inf' if a['margin'] == float('inf') else f"{a['margin']:.2f}"
    flag = '✅' if (a['monotonic'] and a['margin'] >= 1.2) else '⚠️'
    print(f"{flag} {name:14s} 单调={str(a['monotonic']):5s} "
          f"折返={fold:>8s} 余量={marg:>5s} 入射角={a['max_incidence_deg']:.1f}°")
print('\n✅ 练习 3 通过：一条零成本的标定验收项，'
      '而且它对「余量不足」也会给出信号')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def distortion_audit(dist, r_max):
    fold = first_fold(dist, r_hi=max(3.0, 2.5 * r_max))
    mono = (fold is None) or (fold > r_max)      # 只关心画幅内
    margin = float('inf') if fold is None else fold / r_max
    return {'monotonic': bool(mono),
            'fold_at': fold,
            'margin': margin,
            'max_incidence_deg': float(np.rad2deg(np.arctan(r_max)))}

a = distortion_audit(DIST, R_MAX)
assert a['monotonic'] is True and a['margin'] > 2.0
assert 1.0 < distortion_audit((-0.28,0,0,0,0), R_MAX)['margin'] < 1.2
assert distortion_audit((-0.40, 0.60, -1.20, 0, 0), R_MAX)['monotonic'] is False
print('✅ 参考答案 3 通过')
print('   monotonic 只关心 [0, r_max]；margin 看折返点离画幅角有多远。'
      '两者都要报——「画幅内单调但余量 1.19」是一个真实的风险状态。')

## ✏️ 练习 4：投影链闭环断言

这是本模块最重要的交付物：**唯一能抓住「静默地错 2 倍」的检查。**

实现 `closure_ok(Kmat, pitch=0.0, tol_m=1e-6)`：
用给定的 `Kmat` 把若干**已知的地面点**投影到像素，再用同一个 `Kmat` 反投影回地面，
断言与原坐标的偏差小于 `tol_m`。返回 `(bool, 最大偏差)`。

然后用它检查第 5 节那四种情形。

In [ ]:
def closure_ok(Kmat, pitch=0.0, tol_m=1e-6, dist=(0,0,0,0,0)):
    """返回 (是否闭环, 最大偏差 m)。地面点用 z=0 的 ground 假设。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
GROUND_PTS = [(10., 1.75, 0.), (20., 0., 0.), (30., -3.2, 0.), (60., 1.75, 0.)]

ok, dev = closure_ok(K)
assert ok and dev < 1e-6, (ok, dev)
print(f'原始 K：闭环 {ok}，最大偏差 {dev:.2e} m')

K_half, _ = adjust_K(K, 'resize', out_w=960, out_h=540)
ok, dev = closure_ok(K_half)
assert ok, '正确同步后的 K 必须闭环'
print(f'正确同步的 K_half：闭环 {ok}，最大偏差 {dev:.2e} m')

# 三种错法都必须被抓住
BROKEN = {
    '两个都忘了改':     K.copy(),
    '只改焦距、忘主点': np.array([[600,0,CX],[0,600,CY],[0,0,1.]]),
    '只改主点、忘焦距': np.array([[F,0,480.],[0,F,270.],[0,0,1.]]),
}
for name, Kb in BROKEN.items():
    # 用错的 K 去解「缩放后的像素」——模拟真实的不一致
    ok_b, dev_b = closure_ok(Kb)
    print(f'  {name:20s} 闭环={ok_b}  最大偏差={dev_b:.3e} m')

# 关键断言：'只改主点、忘焦距' 这一种，单看闭环**是过的**（因为它自洽），
# 必须靠「与另一个 K 的交叉检查」才能抓住 —— 这就是下面的 cross_check
def cross_check(K_train, K_deploy, tol_m=1e-6):
    '''同一个世界点，用两套 K 各走一遍链，读出的米数应当一致。'''
    worst = 0.0
    for P in GROUND_PTS:
        uv_t = (K_train @ np.array([P[1]/P[0]*-1, (H-P[2])/P[0], 1.]))[:2]
        d_t = H * K_train[1,1] / (uv_t[1] - K_train[1,2])
        d_d = H * K_deploy[1,1] / (uv_t[1] - K_deploy[1,2])
        worst = max(worst, abs(d_t - d_d))
    return worst < tol_m, worst

same, w = cross_check(K, K)
assert same and w < 1e-9
bad, w = cross_check(K, BROKEN['只改主点、忘焦距'])
assert not bad, '训练/部署两套 K 不一致必须被抓住'
print(f'\ncross_check(K, 只改主点忘焦距) -> 不一致，最大差 {w:.2f} m')
print('✅ 练习 4 通过：**闭环抓自洽性，交叉检查抓两侧一致性——两个都要有**')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def closure_ok(Kmat, pitch=0.0, tol_m=1e-6, dist=(0,0,0,0,0)):
    worst = 0.0
    for P in GROUND_PTS:
        st = project_steps(P, pitch=pitch, dist=dist, Kmat=Kmat)
        if st is None:
            return False, float('inf')
        u, v = st['uv']
        # 反投影：用同一个 Kmat 回到归一化平面，再按 z=0 求交
        xn = (u - Kmat[0, 2]) / Kmat[0, 0]
        yn = (v - Kmat[1, 2]) / Kmat[1, 1]
        d_v = R_vc(pitch) @ np.array([xn, yn, 1.0])
        if d_v[2] >= -1e-12:
            return False, float('inf')
        s = (0.0 - H) / d_v[2]
        back = CAM_T + s * d_v
        worst = max(worst, float(np.max(np.abs(back - np.array(P)))))
    return worst < tol_m, worst

ok, dev = closure_ok(K)
assert ok and dev < 1e-6
print('✅ 参考答案 4 通过')
print('   注意：闭环检查用的是**同一个** K 的一致性，'
      '所以它抓不住「训练与部署各自自洽但互不相同」——那需要 cross_check')

## 🧪 真实工程胶囊

```python
# ── 1) OpenCV 的对应函数（本课自己实现了一遍，为了看清误差来源）──
import cv2
K   = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]], np.float64)
dist = np.array([k1,k2,p1,p2,k3], np.float64)   # ⚠️ 顺序是 k1,k2,p1,p2,k3
xy_n = cv2.undistortPoints(uv.reshape(-1,1,2), K, dist)   # -> 归一化平面
map1, map2 = cv2.initUndistortRectifyMap(K, dist, None, K_new, size, cv2.CV_32FC1)
img_u = cv2.remap(img, map1, map2, cv2.INTER_LINEAR)      # ← 这就是第 5 节的 LUT

# ── 2) 把 K 与图像尺寸绑在一起，让「忘记同步」变成类型错误 ──
@dataclass(frozen=True)
class CalibratedImage:
    img: np.ndarray
    K: np.ndarray
    dist: np.ndarray
    calib_sha: str            # ← LUT / K 的来源指纹，见第 5 节的警告

def resize(ci: CalibratedImage, out_w, out_h) -> CalibratedImage:
    s = np.array([[out_w/ci.img.shape[1], 0, 0],
                  [0, out_h/ci.img.shape[0], 0], [0, 0, 1]])
    return CalibratedImage(cv2.resize(ci.img, (out_w, out_h)),
                           s @ ci.K, ci.dist, ci.calib_sha)
#   ↑ 关键：**没有只返回图像的 resize**。想拿到缩放后的图，就必须拿到新的 K。

# ── 3) 进 CI 的两条断言（练习 3 与练习 4）──
def test_distortion_in_range():
    a = distortion_audit(dist, r_max=np.hypot(cx, cy)/fx)
    assert a['monotonic'] and a['margin'] >= 1.2, a

def test_pipeline_closure():
    for op in TRAIN_PREPROCESS_OPS:        # 与训练侧共用同一份配置
        K2, _ = adjust_K(K, **op)
        ok, dev = closure_ok(K2)
        assert ok, f'{op} 之后闭环失败，偏差 {dev} m'
```

> **落地顺序建议**：先加 `test_pipeline_closure`（20 行、抓一整类静默错误），
> 再把 `resize` 改成返回 `(img, K)` 的形式（改动大但一次性），
> 最后才是精度层面的 LUT 与自适应迭代。